# Notebook 5: Model Evaluation & Early Warning Comparison
## AI-Powered Smart University Assistant

This notebook covers:
- Full evaluation metrics: Accuracy, Precision, Recall, F1, ROC-AUC, Confusion Matrix
- ROC curves for all models
- Early warning model (30-day data) vs full-semester comparison
- Discussion of key findings

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle, sys, os
sys.path.append('..')
warnings = __import__('warnings'); warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                              roc_auc_score, confusion_matrix, roc_curve, ConfusionMatrixDisplay)
print('Libraries loaded.')

## 1. Load Data and Best Model

In [ ]:
df = pd.read_csv('../data/processed/student_features.csv')
drop_cols = [c for c in ['id_student', 'code_module', 'code_presentation', 'final_result', 'At_Risk'] if c in df.columns]
X = df.drop(columns=drop_cols)
y = df['At_Risk']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

with open('../models/risk_prediction_model.pkl', 'rb') as f:
    best_model = pickle.load(f)
print('Best model loaded.')

## 2. Full Evaluation Report

In [ ]:
from sklearn.metrics import classification_report

y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

print('=== Full Evaluation Report ===')
print(f'Accuracy:  {accuracy_score(y_test, y_pred):.4f}')
print(f'Precision: {precision_score(y_test, y_pred):.4f}')
print(f'Recall:    {recall_score(y_test, y_pred):.4f}')
print(f'F1-Score:  {f1_score(y_test, y_pred):.4f}')
print(f'ROC-AUC:   {roc_auc_score(y_test, y_proba):.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['Not At Risk', 'At Risk']))

## 3. Confusion Matrix

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Not At Risk', 'At Risk'])
disp.plot(ax=ax, cmap='Blues')
ax.set_title('Confusion Matrix — Best Model', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../screenshots/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'True Negatives: {tn} | False Positives: {fp}')
print(f'False Negatives: {fn} | True Positives: {tp}')

## 4. ROC Curve

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_proba)
auc = roc_auc_score(y_test, y_proba)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(fpr, tpr, color='#6366f1', lw=2, label=f'Best Model (AUC = {auc:.3f})')
ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random Classifier')
ax.fill_between(fpr, tpr, alpha=0.1, color='#6366f1')
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve — Academic Risk Prediction', fontsize=13, fontweight='bold')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig('../screenshots/roc_curve.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Early Warning Model (30-day) Comparison

In [ ]:
early_df = pd.read_csv('../data/processed/early_warning_features.csv')
early_drop = [c for c in ['id_student', 'code_module', 'code_presentation', 'final_result', 'At_Risk'] if c in early_df.columns]
X_early = early_df.drop(columns=early_drop)
y_early = early_df['At_Risk']

X_etrain, X_etest, y_etrain, y_etest = train_test_split(X_early, y_early, test_size=0.2, random_state=42, stratify=y_early)

with open('../models/early_risk_prediction_model.pkl', 'rb') as f:
    early_model = pickle.load(f)

y_epred = early_model.predict(X_etest)
y_eproba = early_model.predict_proba(X_etest)[:, 1]

comparison = pd.DataFrame({
    'Full Semester Model': {
        'Accuracy': accuracy_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1': f1_score(y_test, y_pred),
        'ROC-AUC': roc_auc_score(y_test, y_proba)
    },
    'Early Warning (30-day)': {
        'Accuracy': accuracy_score(y_etest, y_epred),
        'Recall': recall_score(y_etest, y_epred, zero_division=0),
        'F1': f1_score(y_etest, y_epred, zero_division=0),
        'ROC-AUC': roc_auc_score(y_etest, y_eproba) if len(np.unique(y_etest)) > 1 else 0
    }
}).round(4)

print('=== Full vs Early Warning Model Comparison ===')
print(comparison.to_string())

## 6. Early Warning Discussion

The early warning model uses only data from the **first 30 days** of the course (before most assessments are complete).

**Key Observations:**
- The early warning model naturally has **lower accuracy and recall** since it has fewer features
- Despite having less data, it can still identify a significant proportion of at-risk students
- The main early predictors are: VLE activity in the first 2 weeks, and initial assessment performance

**Practical Use for Advisors:**
- The early warning system can flag students in **week 4-5** for preventive check-ins
- This is more valuable than waiting until the end of semester when it's too late
- Advisors should use these predictions as a **support tool, not a final judgment**